# Этап 2 — Очистка данных и расчёт RFM

**Цель:** применить решения по очистке, принятые на этапе 1 (EDA), и посчитать три метрики поведения на каждого клиента — **RFM** (Recency, Frequency, Monetary). Результат — таблица «клиент → RFM», основа для кластеризации (этап 3) и предсказания оттока (этап 4).

**Что делаем:**
1. **Очистка:** удаляем строки без `Customer ID` и с `Price = 0`. Полное обоснование каждого решения — в `01_eda.ipynb` (там же разбор пересечений проблем).
2. **RFM:** считаем через функцию `compute_rfm` из `src/features.py` — та же функция переиспользуется на этапе 4 (принцип DRY: логика в одном месте).
3. **Сохраняем** очищенные транзакции и таблицу RFM для следующих этапов.

In [1]:
# импорт нужных библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_excel("../data/raw/online_retail_II.xlsx")
df.shape

(525461, 8)

In [3]:
df_clean = df.dropna(subset=['Customer ID'])
df_clean.shape

(417534, 8)

In [4]:
df_clean = df_clean[df_clean['Price'] > 0]
df_clean.shape

(417503, 8)

In [5]:
df_clean.to_csv('../data/processed/transactions_clean.csv', index=False)

## Очистка данных

Исходный датасет: 525 461 строк

После удаления NaN в Customer ID: 417 534 строк (-107 927)

После удаления Price = 0: 417 503 строк (-31)

Итого для анализа: 417 503 строк

In [6]:
df_clean['InvoiceDate'].max()

Timestamp('2010-12-09 20:01:00')

In [7]:
from datetime import timedelta
reference_date = df_clean['InvoiceDate'].max() + timedelta(days=1)
reference_date

Timestamp('2010-12-10 20:01:00')

### RFM-метрики

Три числа на каждого клиента, описывающие его покупательское поведение:

- **Recency** — сколько дней прошло с последней покупки. Меньше = недавно активен. Считается относительно **опорной даты** = `max(InvoiceDate) + 1 день` (момент «снимка» базы; +1 день, чтобы у купившего в последний день Recency был 1, а не 0).
- **Frequency** — число уникальных заказов (`Invoice`) клиента. Больше = чаще покупает.
- **Monetary** — суммарная выручка от клиента (`Price × Quantity`, с учётом возвратов — они уходят в минус).

Считаем функцией `compute_rfm(data, reference_date)`. Внутри она также **отсекает клиентов с Monetary ≤ 0** — это те, у кого в выборке только возвраты без покупок (артефакт данных, а не реальное поведение).

In [8]:
import sys
sys.path.append('../src')     
from features import compute_rfm


rfm = compute_rfm(df_clean,reference_date)


In [9]:
print(rfm.shape)
print(rfm.head())

(4283, 3)
             Recency  Frequency  Monetary
Customer ID                              
12347.0            3          2   1323.32
12348.0           74          1    222.16
12349.0           43          4   2646.99
12351.0           11          1    300.93
12352.0           11          2    343.80


In [10]:
rfm.to_csv('../data/processed/rfm.csv')

## Итог этапа 2

- **Очистка:** 525 461 → **417 503** строк (удалены пропуски `Customer ID` и `Price = 0`; детали и обоснование — в EDA).
- **Возвраты клиентов** (`Quantity < 0` с валидным `Customer ID`) сохранены — их отрицательный вклад корректно снижает `Monetary`, отражая реальную ценность клиента.
- **RFM посчитан на 4283 клиента** (после отсева `Monetary ≤ 0`).

**Сохранено:**
- `data/processed/transactions_clean.csv` — очищенные транзакции (понадобятся на этапе 4 для временного сплита);
- `data/processed/rfm.csv` — RFM на клиента (вход для кластеризации).

Дальше — **этап 3**: нормализация RFM (StandardScaler) и кластеризация KMeans.